To get the path of data.

In [1]:
import os
print(os.getcwd())


/home/jovyan/work/data/data1


In [2]:
from pyspark.sql import SparkSession

# إنشاء SparkSession
spark = SparkSession.builder.appName("ReadSingleJSON").getOrCreate()

# تحديد المسار الكامل للملف المحدد
json_file_path = "/home/jovyan/work/data/data1/file2.json"

# قراءة الملف فقط
df = spark.read.json(json_file_path)

# عرض البيانات
df.show()
df.printSchema()


+--------------------+-------------+-------------+-------------+
|         restaurants|results_found|results_shown|results_start|
+--------------------+-------------+-------------+-------------+
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{16668008}, 10...|      1263908|           20|           21|
|[{{{9417}, 10a35f...|          249|           20|           21|
|[{{{313256}, 10a3...|        17151|           20|           21|
|[{{{313256}, 10a3...|        17151|           20|           21|
|[{{{313256}, 10a3...|        17151|           20|           21|
|[{{{313256}, 10a3...|        17151|           20|           21|
|[{{{313256}, 10a3...|   

In [3]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)


In [4]:
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import StructType, ArrayType

def fully_flatten(df, prefix=""):
    while True:
        complex_fields = [
            (field.name, field.dataType)
            for field in df.schema.fields
            if isinstance(field.dataType, (StructType, ArrayType))
        ]
        if not complex_fields:
            break

        for field_name, data_type in complex_fields:
            new_prefix = f"{prefix}{field_name}_" if prefix else f"{field_name}_"
            if isinstance(data_type, StructType):
                subfields = [
                    col(f"{field_name}.{subfield.name}").alias(f"{new_prefix}{subfield.name}")
                    for subfield in data_type.fields
                ]
                df = df.select(
                    *[col(c) for c in df.columns if c != field_name],
                    *subfields
                )
            elif isinstance(data_type, ArrayType):
                df = df.withColumn(field_name, explode_outer(col(field_name)))
    return df

# 1. انفجر restaurants فقط
df_restaurants = df.withColumn("restaurant", explode_outer("restaurants")).select("restaurant.*")

# 2. فك كل التداخلات لأي Struct أو Array جوا restaurant فقط
df_fully_flat = fully_flatten(df_restaurants, prefix="restaurant_")


df_fully_flat.printSchema()


root
 |-- restaurant_restaurant_apikey: string (nullable = true)
 |-- restaurant_restaurant_average_cost_for_two: long (nullable = true)
 |-- restaurant_restaurant_book_url: string (nullable = true)
 |-- restaurant_restaurant_cuisines: string (nullable = true)
 |-- restaurant_restaurant_currency: string (nullable = true)
 |-- restaurant_restaurant_deeplink: string (nullable = true)
 |-- restaurant_restaurant_establishment_types: string (nullable = true)
 |-- restaurant_restaurant_events_url: string (nullable = true)
 |-- restaurant_restaurant_featured_image: string (nullable = true)
 |-- restaurant_restaurant_has_online_delivery: long (nullable = true)
 |-- restaurant_restaurant_has_table_booking: long (nullable = true)
 |-- restaurant_restaurant_id: string (nullable = true)
 |-- restaurant_restaurant_is_delivering_now: long (nullable = true)
 |-- restaurant_restaurant_menu_url: string (nullable = true)
 |-- restaurant_restaurant_name: string (nullable = true)
 |-- restaurant_restauran

In [6]:
df_fully_flat.limit(10)

restaurant_restaurant_apikey,restaurant_restaurant_average_cost_for_two,restaurant_restaurant_book_url,restaurant_restaurant_cuisines,restaurant_restaurant_currency,restaurant_restaurant_deeplink,restaurant_restaurant_establishment_types,restaurant_restaurant_events_url,restaurant_restaurant_featured_image,restaurant_restaurant_has_online_delivery,restaurant_restaurant_has_table_booking,restaurant_restaurant_id,restaurant_restaurant_is_delivering_now,restaurant_restaurant_menu_url,restaurant_restaurant_name,restaurant_restaurant_order_deeplink,restaurant_restaurant_order_url,restaurant_restaurant_photos_url,restaurant_restaurant_price_range,restaurant_restaurant_switch_to_order_menu,restaurant_restaurant_thumb,restaurant_restaurant_url,restaurant_restaurant_restaurant_R_res_id,restaurant_restaurant_restaurant_location_address,restaurant_restaurant_restaurant_location_city,restaurant_restaurant_restaurant_location_city_id,restaurant_restaurant_restaurant_location_country_id,restaurant_restaurant_restaurant_location_latitude,restaurant_restaurant_restaurant_location_locality,restaurant_restaurant_restaurant_location_locality_verbose,restaurant_restaurant_restaurant_location_longitude,restaurant_restaurant_restaurant_location_zipcode,restaurant_restaurant_restaurant_user_rating_aggregate_rating,restaurant_restaurant_restaurant_user_rating_rating_color,restaurant_restaurant_restaurant_user_rating_rating_text,restaurant_restaurant_restaurant_user_rating_votes,restaurant_restaurant_restaurant_restaurant_offers_offer_added_by,restaurant_restaurant_restaurant_restaurant_offers_offer_applicable_on,restaurant_restaurant_restaurant_restaurant_offers_offer_date_added,restaurant_restaurant_restaurant_restaurant_offers_offer_disclaimer,restaurant_restaurant_restaurant_restaurant_offers_offer_end_date,restaurant_restaurant_restaurant_restaurant_offers_offer_friendly_end_date,restaurant_restaurant_restaurant_restaurant_offers_offer_friendly_start_date,restaurant_restaurant_restaurant_restaurant_offers_offer_impressions,restaurant_restaurant_restaurant_restaurant_offers_offer_is_active,restaurant_restaurant_restaurant_restaurant_offers_offer_is_editable,restaurant_restaurant_restaurant_restaurant_offers_offer_is_valid,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_id,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_text,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_type,restaurant_restaurant_restaurant_restaurant_offers_offer_restaurant_list,restaurant_restaurant_restaurant_restaurant_offers_offer_restaurants,restaurant_restaurant_restaurant_restaurant_offers_offer_share_url,restaurant_restaurant_restaurant_restaurant_offers_offer_start_date,restaurant_restaurant_restaurant_restaurant_offers_offer_status,restaurant_restaurant_restaurant_restaurant_offers_offer_type,restaurant_restaurant_restaurant_restaurant_offers_offer_type_code,restaurant_restaurant_restaurant_restaurant_offers_offer_voucher_id,restaurant_restaurant_restaurant_restaurant_zomato_events_event_book_link,restaurant_restaurant_restaurant_restaurant_zomato_events_event_date_added,restaurant_restaurant_restaurant_restaurant_zomato_events_event_description,restaurant_restaurant_restaurant_restaurant_zomato_events_event_disclaimer,restaurant_restaurant_restaurant_restaurant_zomato_events_event_display_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_display_time,restaurant_restaurant_restaurant_restaurant_zomato_events_event_end_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_end_time,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_category,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_category_name,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_id,restaurant_restaurant_restaurant_restaurant_zomato_events_event_friendly_end_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_friendly_start_date,restaurant_restau

In [7]:
print("قبل التفجير:", df_fully_flat.count())
print("بعد التفجير:", df_fully_flat.count())


قبل التفجير: 12260
بعد التفجير: 12260


In [8]:
print("عدد الصفوف الأصلي:", df_fully_flat.count())


عدد الصفوف الأصلي: 12260


In [9]:

print("column names:", df_fully_flat.columns)




column names: ['restaurant_restaurant_apikey', 'restaurant_restaurant_average_cost_for_two', 'restaurant_restaurant_book_url', 'restaurant_restaurant_cuisines', 'restaurant_restaurant_currency', 'restaurant_restaurant_deeplink', 'restaurant_restaurant_establishment_types', 'restaurant_restaurant_events_url', 'restaurant_restaurant_featured_image', 'restaurant_restaurant_has_online_delivery', 'restaurant_restaurant_has_table_booking', 'restaurant_restaurant_id', 'restaurant_restaurant_is_delivering_now', 'restaurant_restaurant_menu_url', 'restaurant_restaurant_name', 'restaurant_restaurant_order_deeplink', 'restaurant_restaurant_order_url', 'restaurant_restaurant_photos_url', 'restaurant_restaurant_price_range', 'restaurant_restaurant_switch_to_order_menu', 'restaurant_restaurant_thumb', 'restaurant_restaurant_url', 'restaurant_restaurant_restaurant_R_res_id', 'restaurant_restaurant_restaurant_location_address', 'restaurant_restaurant_restaurant_location_city', 'restaurant_restaurant_re

In [11]:
import re

def clean_column_name(col_name):
    # يشيل كل تكرار لعبارة restaurant_ في بداية الاسم
    return re.sub(r'^(restaurant_)+', '', col_name)

# إعادة تسمية الأعمدة في DataFrame
for old_name in df_fully_flat.columns:
    new_name = clean_column_name(old_name)
    if new_name != old_name:
        df_fully_flat = df_fully_flat.withColumnRenamed(old_name, new_name)

print("column names:", df_fully_flat.columns)


column names: ['apikey', 'average_cost_for_two', 'book_url', 'cuisines', 'currency', 'deeplink', 'establishment_types', 'events_url', 'featured_image', 'has_online_delivery', 'has_table_booking', 'id', 'is_delivering_now', 'menu_url', 'name', 'order_deeplink', 'order_url', 'photos_url', 'price_range', 'switch_to_order_menu', 'thumb', 'url', 'R_res_id', 'location_address', 'location_city', 'location_city_id', 'location_country_id', 'location_latitude', 'location_locality', 'location_locality_verbose', 'location_longitude', 'location_zipcode', 'user_rating_aggregate_rating', 'user_rating_rating_color', 'user_rating_rating_text', 'user_rating_votes', 'offers_offer_added_by', 'offers_offer_applicable_on', 'offers_offer_date_added', 'offers_offer_disclaimer', 'offers_offer_end_date', 'offers_offer_friendly_end_date', 'offers_offer_friendly_start_date', 'offers_offer_impressions', 'offers_offer_is_active', 'offers_offer_is_editable', 'offers_offer_is_valid', 'offers_offer_offer_id', 'offers_

In [10]:
from pyspark import StorageLevel

# فحص شامل للبيانات
print("=== information about the data ===")
print(f"TOtal of records: {df_fully_flat.count():,}")
print(f"Total of raws: {len(df_fully_flat.columns)}")

# إزالة التكرارات مع persist
df_deduped = df_fully_flat.dropDuplicates().persist(StorageLevel.MEMORY_AND_DISK)

print("=== information about the data ===")
print(f"TOtal of records: {df_deduped.count():,}")  # سريع!
print(f"Total of raws: {len(df_deduped.columns)}")

# تنظيف الذاكرة عند الانتهاء
df_deduped.unpersist()


=== information about the data ===
TOtal of records: 12,260
Total of raws: 86
=== information about the data ===
TOtal of records: 3,969
Total of raws: 86


restaurant_restaurant_apikey,restaurant_restaurant_average_cost_for_two,restaurant_restaurant_book_url,restaurant_restaurant_cuisines,restaurant_restaurant_currency,restaurant_restaurant_deeplink,restaurant_restaurant_establishment_types,restaurant_restaurant_events_url,restaurant_restaurant_featured_image,restaurant_restaurant_has_online_delivery,restaurant_restaurant_has_table_booking,restaurant_restaurant_id,restaurant_restaurant_is_delivering_now,restaurant_restaurant_menu_url,restaurant_restaurant_name,restaurant_restaurant_order_deeplink,restaurant_restaurant_order_url,restaurant_restaurant_photos_url,restaurant_restaurant_price_range,restaurant_restaurant_switch_to_order_menu,restaurant_restaurant_thumb,restaurant_restaurant_url,restaurant_restaurant_restaurant_R_res_id,restaurant_restaurant_restaurant_location_address,restaurant_restaurant_restaurant_location_city,restaurant_restaurant_restaurant_location_city_id,restaurant_restaurant_restaurant_location_country_id,restaurant_restaurant_restaurant_location_latitude,restaurant_restaurant_restaurant_location_locality,restaurant_restaurant_restaurant_location_locality_verbose,restaurant_restaurant_restaurant_location_longitude,restaurant_restaurant_restaurant_location_zipcode,restaurant_restaurant_restaurant_user_rating_aggregate_rating,restaurant_restaurant_restaurant_user_rating_rating_color,restaurant_restaurant_restaurant_user_rating_rating_text,restaurant_restaurant_restaurant_user_rating_votes,restaurant_restaurant_restaurant_restaurant_offers_offer_added_by,restaurant_restaurant_restaurant_restaurant_offers_offer_applicable_on,restaurant_restaurant_restaurant_restaurant_offers_offer_date_added,restaurant_restaurant_restaurant_restaurant_offers_offer_disclaimer,restaurant_restaurant_restaurant_restaurant_offers_offer_end_date,restaurant_restaurant_restaurant_restaurant_offers_offer_friendly_end_date,restaurant_restaurant_restaurant_restaurant_offers_offer_friendly_start_date,restaurant_restaurant_restaurant_restaurant_offers_offer_impressions,restaurant_restaurant_restaurant_restaurant_offers_offer_is_active,restaurant_restaurant_restaurant_restaurant_offers_offer_is_editable,restaurant_restaurant_restaurant_restaurant_offers_offer_is_valid,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_id,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_text,restaurant_restaurant_restaurant_restaurant_offers_offer_offer_type,restaurant_restaurant_restaurant_restaurant_offers_offer_restaurant_list,restaurant_restaurant_restaurant_restaurant_offers_offer_restaurants,restaurant_restaurant_restaurant_restaurant_offers_offer_share_url,restaurant_restaurant_restaurant_restaurant_offers_offer_start_date,restaurant_restaurant_restaurant_restaurant_offers_offer_status,restaurant_restaurant_restaurant_restaurant_offers_offer_type,restaurant_restaurant_restaurant_restaurant_offers_offer_type_code,restaurant_restaurant_restaurant_restaurant_offers_offer_voucher_id,restaurant_restaurant_restaurant_restaurant_zomato_events_event_book_link,restaurant_restaurant_restaurant_restaurant_zomato_events_event_date_added,restaurant_restaurant_restaurant_restaurant_zomato_events_event_description,restaurant_restaurant_restaurant_restaurant_zomato_events_event_disclaimer,restaurant_restaurant_restaurant_restaurant_zomato_events_event_display_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_display_time,restaurant_restaurant_restaurant_restaurant_zomato_events_event_end_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_end_time,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_category,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_category_name,restaurant_restaurant_restaurant_restaurant_zomato_events_event_event_id,restaurant_restaurant_restaurant_restaurant_zomato_events_event_friendly_end_date,restaurant_restaurant_restaurant_restaurant_zomato_events_event_friendly_start_date,restaurant_restau

In [ ]:
final_result= flat_df_final
final_result.show()

In [ ]:
from google.cloud import bigquery
import os

# تحديد مسار ملف الخدمة
key_path = "/home/jovyan/keys/fooddelivary-456823-44ced47a2164.json"

# إنشاء عميل BigQuery
client = bigquery.Client.from_service_account_json(key_path)

# اسم الجدول اللي هترفع فيه الداتا
table_id = "fooddelivary-456823.food_analytics.final_result_cleaned"

# رفع الداتا فريم إلى BigQuery
job = client.load_table_from_dataframe(final_result.toPandas(), table_id).result()

print("✅ تم رفع final_result إلى BigQuery بنجاح.")


In [ ]:
!pip install google-cloud-bigquery --upgrade

